# 🧠 Model 1: CNN-2D
**Input:** `(batch, n_channels, N_BANDS, 1)` — frequency bands as spatial height  
**Feature output:** 128-D penultimate dense vector → downstream SVM

| Block | Operation |
|-------|-----------|
| 1 | Conv2D(32) → BN → MaxPool → Dropout |
| 2 | Conv2D(64) → BN → MaxPool → Dropout |
| 3 | Conv2D(128) → BN → Dropout |
| — | GlobalAvgPool → Dense(128, ReLU) ← **feature vector** |
| — | Dense(1, sigmoid) ← classification head |


## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import tensorflow as tf

# Set seeds for fully reproducible results across runs
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [3]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or '⚠️ No GPU')

Tue Sep 29 20:18:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install scikit-learn openpyxl -q

In [5]:
import os,gc,time,numpy as np,pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,f1_score,classification_report
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers,models,Input,optimizers as opt_module

BAND_NAMES=["Delta","Theta","Alpha","Beta","Gamma"]

def load_dataset(p):
    df = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    # FIX #11: a handful of windows (~7/691185, likely near-zero-power channels
    # in the relative band-power calc causing 0/0) contain NaN feature values.
    # Left in, these propagate NaN through the whole network from batch 1 on
    # (loss: nan, val_auc stuck at 0.5). Drop them -- too few to matter, and
    # dropping is safer than zero-filling (which would fake a flat signal).
    n_nan = df.isna().any(axis=1).sum()
    if n_nan:
        print(f"    [nan guard] dropping {n_nan} rows with NaN features")
        df = df.dropna().reset_index(drop=True)
    return df
def get_band_columns(df): return {b:[c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}
def prepare_tensors_2d(df):
    bc=get_band_columns(df)
    X=np.stack([df[bc[b]].values for b in BAND_NAMES],axis=1)   # (N,5,n_ch)
    X=X[...,np.newaxis].transpose(0,2,1,3)                       # (N,n_ch,5,1)
    return X, df["label"].values.astype(int)

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-channel/per-band), apply to all splits."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte
def evaluate_with_svm(Xtr, ytr, Xte, yte, max_train=20000):
    # FIX #12: RBF-kernel SVC scales ~O(n^2) to O(n^3) in sample count. With
    # ~700K training rows this never finishes in practical time (confirmed:
    # stuck 3h15min+ with zero progress after the CNN finished). Subsample
    # for the SVM step only -- the 128-D deep feature embedding is what's
    # being evaluated here, and a representative subsample is sufficient for
    # this downstream check without changing the evaluation method itself.
    if len(Xtr) > max_train:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]

    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xte = sc.transform(Xte)

    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)
    svm.fit(Xtr, ytr)
    p = svm.predict(Xte)

    # Safely extract confusion matrix with fixed labels to prevent silent crashes
    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()

    # Calculate clinical metrics
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0   # Seizure recall
    specificity = tn / (tn + fp) if (tn + fp) else 0.0   # No-Seizure recall

    return {
        "accuracy": round(accuracy_score(yte, p), 4),
        "sensitivity": round(sensitivity, 4),
        "specificity": round(specificity, 4),
        "f1_macro": round(f1_score(yte, p, average="macro"), 4),
        "f1_weighted": round(f1_score(yte, p, average="weighted"), 4),
        # Added labels=[0, 1] to enforce correct class mapping
        "report": classification_report(yte, p, labels=[0, 1], target_names=["No Seizure", "Seizure"])
    }
def save_results(row,path):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)
    print("Saved:",row)


In [6]:
# Prevent TF from allocating all GPU VRAM at once, which avoids OOM errors on limited GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

✅ GPU Memory Growth Enabled


In [7]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
DATASET_A  = "/content/drive/MyDrive/epilepsy_outputs/dataset_A.csv"
DATASET_B  = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS    = "/content/drive/MyDrive/epilepsy_outputs/results.csv"
OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
EPOCHS=50; BATCH_SIZE=64; PATIENCE=8; SEED=42


In [8]:
from tensorflow.keras import layers, models, Input, regularizers

# ── Build CNN-2D (Final Optimized Architecture) ───────────────────────────────
def build_cnn2d(n_channels=18, n_bands=5, feature_dim=128, dropout=0.4, l2=1e-4, output_bias=None):
    bias_init = tf.keras.initializers.Constant(output_bias) if output_bias is not None else "zeros"
    inp = Input(shape=(n_channels, n_bands, 1), name="eeg_input")

    # Block 1
    # Use (1,3) kernel to prevent spatial mixing of channels.
    # Add L2 regularization to constrain weight magnitude.
    x = layers.Conv2D(32, (1,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1), padding="same")(x)
    # SpatialDropout2D drops entire 2D feature maps, highly effective for Conv layers
    x = layers.SpatialDropout2D(dropout/2)(x)

    # Block 2
    x = layers.Conv2D(64, (1,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1), padding="same")(x)
    x = layers.SpatialDropout2D(dropout/2)(x)

    # Block 3
    x = layers.Conv2D(128, (1,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.SpatialDropout2D(dropout)(x)

    x = layers.GlobalAveragePooling2D()(x)

    # Feature Extractor Head
    feat = layers.Dense(feature_dim, activation="relu", name="feature_vector",
                        kernel_regularizer=regularizers.l2(l2))(x)
    # Standard Dropout is correct here because 'feat' is a 1D dense vector, not a spatial map
    feat = layers.Dropout(dropout)(feat)

    # Classification Head
    out = layers.Dense(1, activation="sigmoid", name="output", bias_initializer=bias_init)(feat)

    full = models.Model(inp, out, name="CNN2D")
    feat_m = models.Model(inp, feat, name="CNN2D_FeatExtractor")

    return full, feat_m

def compile_cnn2d(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
    return m

model, feat_ext = build_cnn2d()
model.summary()
print("Feature extractor output:", feat_ext.output_shape)

# Clean up this standalone graph before starting the real training loop
del model, feat_ext
tf.keras.backend.clear_session()


Model: "CNN2D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 18, 5, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 18, 5, 32)      │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 5, 32)      │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 9, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d               │ (None, 9, 5, 32)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 9, 5, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 9, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_1             │ (None, 5, 5, 64)       │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 5, 5, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_2             │ (None, 5, 5, 128)      │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,577 (189.75 KB)

 Trainable params: 48,129 (188.00 KB)

 Non-trainable params: 448 (1.75 KB)

Feature extractor output: (None, 128)


In [9]:
# ── Training loop ─────────────────────────────────────────────────────────────
def run_trial(ds_path, ds_label, opt_name, opt_obj):
    print(f"\n{'='*55}\n  CNN2D | Dataset {ds_label} | {opt_name}\n{'='*55}")
    df=load_dataset(ds_path); X,y=prepare_tensors_2d(df)
    n_ch,n_bands=X.shape[1],X.shape[2]
    Xtrv,Xte,ytrv,yte=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
    Xtr,Xv,ytr,yv=train_test_split(Xtrv,ytrv,test_size=0.15/0.85,stratify=ytrv,random_state=SEED)
    Xtr,Xv,Xte=normalize_tensors(Xtr,Xv,Xte)   # train-only normalization
    total=len(ytr); n1=ytr.sum(); cw={0:total/(2*(total-n1)),1:total/(2*n1)}
    output_bias=np.log(n1/(total-n1))          # prior-informed bias init
    full_m,feat_m=build_cnn2d(n_channels=n_ch,n_bands=n_bands,output_bias=output_bias)
    full_m=compile_cnn2d(full_m,opt_obj)
    ckpt=os.path.join(OUTPUT_DIR,"checkpoints",f"CNN2D_{ds_label}_{opt_name}_best.keras")
    os.makedirs(os.path.dirname(ckpt),exist_ok=True)
    cbs=[tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,restore_best_weights=True,verbose=1),
         tf.keras.callbacks.ModelCheckpoint(ckpt,monitor="val_auc",mode="max",save_best_only=True,verbose=0),
         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=4,min_lr=1e-6,verbose=1)]
    t0=time.time()
    hist=full_m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=EPOCHS,batch_size=BATCH_SIZE,
                    class_weight=cw,callbacks=cbs,verbose=1)
    elapsed=round((time.time()-t0)/60,2)
    best_auc=max(hist.history.get("val_auc",[0]))
    print(f"  Done {elapsed}min | Best AUC:{best_auc:.4f}")
    f_tr=feat_m.predict(Xtr,batch_size=BATCH_SIZE,verbose=0)
    f_te=feat_m.predict(Xte,batch_size=BATCH_SIZE,verbose=0)
    res=evaluate_with_svm(f_tr,ytr,f_te,yte)
    print(f"  SVM Acc:{res['accuracy']} F1:{res['f1_macro']}\n",res["report"])
    save_results({"model":"CNN2D","dataset":f"Dataset_{ds_label}","optimizer":opt_name,
                  "best_val_auc":round(best_auc,4),**{k:v for k,v in res.items() if k!="report"},
                  "train_time_min":elapsed}, RESULTS)
    del full_m,feat_m; gc.collect(); tf.keras.backend.clear_session()

for ds_label,ds_path in [("A",DATASET_A),("B",DATASET_B)]:
    for opt_name,opt_obj in {"Adam":opt_module.Adam(1e-3,clipnorm=1.0),
                              "Nadam":opt_module.Nadam(1e-3,clipnorm=1.0),
                              "AdamW":opt_module.AdamW(1e-3,weight_decay=1e-4,clipnorm=1.0)}.items():
        run_trial(ds_path,ds_label,opt_name,opt_obj)
print("\n✅ CNN-2D all trials done.")



  CNN2D | Dataset A | Adam
    [nan guard] dropping 7 rows with NaN features
Epoch 1/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 57s 6ms/step - accuracy: 0.9970 - auc: 0.5649 - loss: 2.1449 - precision: 0.0128 - recall: 0.0061 - val_accuracy: 0.9980 - val_auc: 0.5009 - val_loss: 0.0473 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step - accuracy: 0.9980 - auc: 0.6118 - loss: 1.5342 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9980 - val_auc: 0.3627 - val_loss: 0.1064 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - accuracy: 0.9945 - auc: 0.7468 - loss: 1.1096 - precision: 0.0152 - recall: 0.0265 - val_accuracy: 0.7050 - val_auc: 0.3575 - val_loss: 0.4532 - val_precision: 2.6323e-04 - val_recall: 0.0381 - learning_rate: 0.0010
Epoch 4/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - accuracy: 0.9776 - auc: 0.7606